In [1]:
import pandas as pd
res = pd.read_csv("../data/results.csv", na_values=r"\N")
races = pd.read_csv("../data/races.csv")
print(res.shape, races.shape)
print(res.head())

FileNotFoundError: [Errno 2] No such file or directory: '../data/results.csv'

In [2]:
import os
print(os.getcwd())

C:\Users\Andrii\Documents\f1-grid-advantage


In [3]:
import pandas as pd
res = pd.read_csv("data/results.csv", na_values=r"\N")
races = pd.read_csv("data/races.csv")
print(res.shape, races.shape)
print(res.head())

(26759, 18) (1125, 18)
   resultId  raceId  driverId  constructorId  number  grid  position  \
0         1      18         1              1    22.0     1       1.0   
1         2      18         2              2     3.0     5       2.0   
2         3      18         3              3     7.0     7       3.0   
3         4      18         4              4     5.0    11       4.0   
4         5      18         5              1    23.0     3       5.0   

  positionText  positionOrder  points  laps         time  milliseconds  \
0            1              1    10.0    58  1:34:50.616     5690616.0   
1            2              2     8.0    58       +5.478     5696094.0   
2            3              3     6.0    58       +8.163     5698779.0   
3            4              4     5.0    58      +17.181     5707797.0   
4            5              5     4.0    58      +18.014     5708630.0   

   fastestLap  rank fastestLapTime  fastestLapSpeed  statusId  
0        39.0   2.0       1:27.452 

In [4]:
import numpy as np
import statsmodels.formula.api as smf

df = res.merge(races[["raceId", "year"]], on="raceId")
print("years in data:", df["year"].min(), "-", df["year"].max())

df = df[(df["year"] >= 2014) & (df["grid"] > 0)].copy()
df["era"] = np.where(df["year"] >= 2022, "ground_effect", "hybrid")
df["podium"] = (df["positionOrder"] <= 3).astype(int)

for era, d in df.groupby("era"):
    fin = d.dropna(subset=["position"])
    lin = smf.ols("position ~ grid", data=fin).fit()
    log = smf.logit("podium ~ grid", data=d).fit(disp=0)
    print(era, "| slope:", round(lin.params["grid"], 2),
          "| R2:", round(lin.rsquared, 2),
          "| odds ratio:", round(np.exp(log.params["grid"]), 2))

years in data: 1950 - 2024
ground_effect | slope: 0.67 | R2: 0.53 | odds ratio: 0.62
hybrid | slope: 0.64 | R2: 0.57 | odds ratio: 0.6
